In [1]:
# Use this to create a cumulative NDMI image for the 2025 dry season.
# This was used in the graphical abstract. 
# This image was not directly used in the science, its point was more to demonstrate how 
# the moisture  averaged over the entire dry season can be visible from space, even for small plots.

import importlib
import rasterio
import Utils 
import numpy as np
import Constants
import ConstantObjects
importlib.reload(Utils)
importlib.reload(Constants)
importlib.reload(ConstantObjects)
import inspect
import os
from rasterio.mask import mask

print(f"[Line {inspect.currentframe().f_lineno}] ...  Done with cell. ")

[Line 13] n_cols in ConstantObjects: 21
[Line 15] n_cols: 21, n_rows: 18
[Line 26] n_cols in ConstantObjects: 21
[Line 28] gdf_tree_circles.shape: (378, 1)
x0 =  -10783454.50783975
y0 =  2246774.099946275
draw_grid_box : dx =  5
draw_grid_box : dy =  4
📍 Center of box: (19.777898, -96.869939)
x0 =  -10783454.50783975
y0 =  2246774.099946275
draw_grid_box : dx =  5
draw_grid_box : dy =  4
📍 Center of box: (19.777882, -96.869634)
x0 =  -10783576.959279623
y0 =  2246703.1213188255
draw_grid_box : dx =  5
draw_grid_box : dy =  4
📍 Center of box: (19.777201, -96.871090)
Constants.n_cols =  21
Constants.n_rows =  18
x0 =  -10783454.50783975
y0 =  2246395.5503548854
draw_grid_box : dx =  5
draw_grid_box : dy =  4
📍 Center of box: (19.778652, -96.863761)
x0 =  -10783454.50783975
y0 =  2246774.099946275
draw_grid_box : dx =  5
draw_grid_box : dy =  4
📍 Center of box: (19.777761, -96.869698)
x0 =  -10783398.848094355
y0 =  2246927.887889348
draw_grid_box : dx =  5
draw_grid_box : dy =  4
📍 Cente

In [8]:
import pandas as pd
import importlib
import rasterio
import Utils 
import numpy as np
import Constants
import ConstantObjects
importlib.reload(Utils)
importlib.reload(Constants)
importlib.reload(ConstantObjects)
import inspect
import os
from rasterio.mask import mask
from IPython.display import display, HTML
display(HTML("<style>.jp-OutputArea {max-height: 200px}</style>"))

# path list to hold all NDMI rasters
ndmi_paths = []

#Some Sentinel2 overflight dates in veracruz are 2024-01-13, 2025-01-12
startDate = "2025-03-03"
endDate = "2025-06-30"

# Villanueva, VER
#startDate = "2025-01-12"
#endDate = "2025-08-16"
#ethiopia
#startDate = "2021-10-01"
#endDate = "2022-06-30"


dates = Utils.generate_date_range(startDate, endDate,  "%Y-%m-%d", 5)
print(f"[Line {inspect.currentframe().f_lineno}] ...   ")

# jan 12, no cloud directly over farm.
#Feb 11, lots of clouds over farm
ndmi_tile_tif=""
print(f"Cell [Line {inspect.currentframe().f_lineno}] ... dates = ",dates)
        
for date in dates:
    
    print(f"Cell [Line {inspect.currentframe().f_lineno}] ... date = ",date)
    gndvi_tile_tif = Utils.make_path_name("s2_derived_tifs", Utils.latlon_to_s2_tile(Constants.lat0, Constants.lon0),date, "gndvi","tif")
    ndmi_tile_tif  = Utils.make_path_name("s2_derived_tifs", Utils.latlon_to_s2_tile(Constants.lat0, Constants.lon0),date, "ndmi", "tif")

    ndmi_paths.append(ndmi_tile_tif)
    rgb_tile_tif   = Utils.make_path_name("s2_derived_tifs", Utils.latlon_to_s2_tile(Constants.lat0, Constants.lon0),date, "rgb.uint8","tif")

    print(f"Cell [Line {inspect.currentframe().f_lineno}] ... ndmi_tile_tif = ",ndmi_tile_tif)
    gdf_box = ConstantObjects.gdf_box_terreno_casa
    ndmi_terreno_casa_tif = Utils.make_path_name("s2_parcel_tifs", Utils.latlon_to_s2_tile(Constants.lat0, Constants.lon0),date, "ndmi-terreno-casa","tif")
    print(f"Cell [Line {inspect.currentframe().f_lineno}] ... ndmi_terreno_casa_tif = ",ndmi_terreno_casa_tif)
    ndmi_plots_2_3_1A_1B_tif  = Utils.make_path_name("s2_parcel_tifs", Utils.latlon_to_s2_tile(Constants.lat0, Constants.lon0),date, "plots_2_3_1A_1B","tif")
    ndmi_casa_y_vecino_tif  = Utils.make_path_name("s2_parcel_tifs", Utils.latlon_to_s2_tile(Constants.lat0, Constants.lon0),date, "casa_y_vecino","tif")

    b02_terreno_casa_jp2 = Utils.make_path_name("s2_point_series", Utils.latlon_to_s2_tile(Constants.lat0, Constants.lon0),date, "B02","jp2")
    b03_terreno_casa_jp2 = Utils.make_path_name("s2_point_series", Utils.latlon_to_s2_tile(Constants.lat0, Constants.lon0),date, "B03","jp2")
    b04_terreno_casa_jp2 = Utils.make_path_name("s2_point_series", Utils.latlon_to_s2_tile(Constants.lat0, Constants.lon0),date, "B04","jp2")
    if os.path.exists(b02_terreno_casa_jp2) and os.path.exists(b03_terreno_casa_jp2) and os.path.exists(b04_terreno_casa_jp2):
        print(f"[Line {inspect.currentframe().f_lineno}] ... found all three of B02,B03, and B04 .jp2 images. Moving forward. ")
    else:
        print(f"[Line {inspect.currentframe().f_lineno}] ... will have to try the next date, because unable to find files:  ", b02_terreno_casa_jp2 ," , " , b03_terreno_casa_jp2," , " , b04_terreno_casa_jp2  )
        continue #If we do not have all three files for this date we should move on to the next date.
    green_vs_red_blue = np.nan
    rgb_terreno_casa_tif = Utils.make_path_name("s2_parcel_tifs", Utils.latlon_to_s2_tile(Constants.lat0, Constants.lon0),date, "rgb-terreno-casa","tif")

    print(f"[Line {inspect.currentframe().f_lineno}] ... ndmi_terreno_casa_tif = ",ndmi_terreno_casa_tif)    
    # add up red intensity
    b2_intensity_terreno_casa = Utils.crop_and_sum_intensity(b02_terreno_casa_jp2, ConstantObjects.gdf_box_terreno_casa, scale_factor=None, clip_to_aoi_bounds=True)
    b3_intensity_terreno_casa = Utils.crop_and_sum_intensity(b03_terreno_casa_jp2, ConstantObjects.gdf_box_terreno_casa, scale_factor=None, clip_to_aoi_bounds=True)
    b4_intensity_terreno_casa = Utils.crop_and_sum_intensity(b04_terreno_casa_jp2, ConstantObjects.gdf_box_terreno_casa, scale_factor=None, clip_to_aoi_bounds=True)
    b2_norm_reflectance_terreno_casa = b2_intensity_terreno_casa["sum"]/b2_intensity_terreno_casa["valid_pixels"]
    b3_norm_reflectance_terreno_casa = b3_intensity_terreno_casa["sum"]/b3_intensity_terreno_casa["valid_pixels"]
    b4_norm_reflectance_terreno_casa = b4_intensity_terreno_casa["sum"]/b4_intensity_terreno_casa["valid_pixels"]
    red_green_blue_average = (b2_norm_reflectance_terreno_casa+b3_norm_reflectance_terreno_casa+b3_norm_reflectance_terreno_casa)/3 
    cloudy =0
    if (red_green_blue_average) > 1650:  cloudy = 1

    print(f"[Line {inspect.currentframe().f_lineno}] ... calling Utils.crop_ndmi_to_gdf(ndmi_tile_tif, ConstantObjects.gdf_box_terreno_casa, ndmi_terreno_casa_tif) ")
    stats_terreno_casa = {
        "count_valid": 0,
        "mean": float("nan"),
        "median": float("nan"),
    }
    stats_casa_y_vecino = stats_terreno_casa

    print(f"Cell [Line {inspect.currentframe().f_lineno}] ... ")
    if not cloudy: 
        print(f"Cell [Line {inspect.currentframe().f_lineno}] ... cloudy = ",cloudy)
        
        #stats_casa_y_vecino  = Utils.crop_ndmi_to_gdf(ndmi_tile_tif, ConstantObjects.gdf_box_casa_y_vecino , ndmi_casa_y_vecino_tif)  

    print(f"Cell [Line {inspect.currentframe().f_lineno}] ... ")

  
        

    print(f"Cell [Line {inspect.currentframe().f_lineno}] ... date  ",date)
  

  
    print(f"[Line {inspect.currentframe().f_lineno}] ... Just appended to rows, an entry with date = ",date)
    print(f"[Cell Line {inspect.currentframe().f_lineno}] ... b2_intensity_terreno_casa[sum]  ",b2_intensity_terreno_casa["sum"])
    print(f"Cell [Line {inspect.currentframe().f_lineno}] ... b3_intensity_terreno_casa[sum]  ",b3_intensity_terreno_casa["sum"])
    print(f"Cell [Line {inspect.currentframe().f_lineno}] ... b4_intensity_terreno_casa[sum]  ",b4_intensity_terreno_casa["sum"])
    #print(f"Cell [Line {inspect.currentframe().f_lineno}] ... stats_milpa_vecino.get(mean,   np.nan)  ",stats_milpa_vecino.get("mean",   np.nan))

    print(f"[Line {inspect.currentframe().f_lineno}] ... calling Utils.crop_ndmi_to_gdf(rgb_tile_tif, ConstantObjects.gdf_box_terreno_casa, rgb_terreno_casa_tif)  ")
 
    print(f"[Line {inspect.currentframe().f_lineno}] ... ")

print(f"[Cell Line {inspect.currentframe().f_lineno}] .. Done with cell! ")



[Line 13] n_cols in ConstantObjects: 21
[Line 15] n_cols: 21, n_rows: 18
[Line 26] n_cols in ConstantObjects: 21
[Line 28] gdf_tree_circles.shape: (378, 1)
x0 =  -10783454.50783975
y0 =  2246774.099946275
draw_grid_box : dx =  5
draw_grid_box : dy =  4
📍 Center of box: (19.777898, -96.869939)
x0 =  -10783454.50783975
y0 =  2246774.099946275
draw_grid_box : dx =  5
draw_grid_box : dy =  4
📍 Center of box: (19.777882, -96.869634)
x0 =  -10783576.959279623
y0 =  2246703.1213188255
draw_grid_box : dx =  5
draw_grid_box : dy =  4
📍 Center of box: (19.777201, -96.871090)
Constants.n_cols =  21
Constants.n_rows =  18
x0 =  -10783454.50783975
y0 =  2246395.5503548854
draw_grid_box : dx =  5
draw_grid_box : dy =  4
📍 Center of box: (19.778652, -96.863761)
x0 =  -10783454.50783975
y0 =  2246774.099946275
draw_grid_box : dx =  5
draw_grid_box : dy =  4
📍 Center of box: (19.777761, -96.869698)
x0 =  -10783398.848094355
y0 =  2246927.887889348
draw_grid_box : dx =  5
draw_grid_box : dy =  4
📍 Cente

[Line 33] ...   
Cell [Line 38] ... dates =  ['2025-03-03', '2025-03-08', '2025-03-13', '2025-03-18', '2025-03-23', '2025-03-28', '2025-04-02', '2025-04-07', '2025-04-12', '2025-04-17', '2025-04-22', '2025-04-27', '2025-05-02', '2025-05-07', '2025-05-12', '2025-05-17', '2025-05-22', '2025-05-27', '2025-06-01', '2025-06-06', '2025-06-11', '2025-06-16', '2025-06-21', '2025-06-26']
Cell [Line 42] ... date =  2025-03-03
[Line 124] ... found mgrs_tile =  14QQG
band = > gndvi <
extension = > tif <
[Line 124] ... found mgrs_tile =  14QQG
band = > ndmi <
extension = > tif <
[Line 124] ... found mgrs_tile =  14QQG
band = > rgb.uint8 <
extension = > tif <
Cell [Line 49] ... ndmi_tile_tif =  s2_derived_tifs/14QQG_2025-03-03_ndmi.tif
[Line 124] ... found mgrs_tile =  14QQG
band = > ndmi-terreno-casa <
extension = > tif <
Cell [Line 52] ... ndmi_terreno_casa_tif =  s2_parcel_tifs/14QQG_2025-03-03_ndmi-terreno-casa.tif
[Line 124] ... found mgrs_tile =  14QQG
band = > plots_2_3_1A_1B <
extension = > 

In [10]:



# ---- Compute cumulative average NDMI ----

sum_arr = None
count = 0
meta = None

for i, path in enumerate(ndmi_paths):
    print(f"[Line {inspect.currentframe().f_lineno}] ... processing : ",path)
    with rasterio.open(path) as src:
        arr = src.read(1).astype("float32") # read NDMI band
        # store metadata from first file
        if meta is None:
            meta = src.meta.copy()
        # initialize accumulator
        if sum_arr is None:
            sum_arr = np.zeros_like(arr, dtype="float32")
        # accumulate sum (ignore nodata values)
        mask = arr != src.nodata
        sum_arr[mask] += arr[mask]
        count += 1

# Compute mean
mean_arr = sum_arr / count

# Prepare metadata for output file
meta.update(dtype="float32")
output_file = "ndmi_mean.tif"

# Save cumulative mean GeoTIFF
with rasterio.open(output_file, "w", **meta) as dst:
    dst.write(mean_arr.astype("float32"), 1)
print("Cumulative averaged NDMI saved to:", output_file)

[Line 8] ... processing :  s2_derived_tifs/14QQG_2025-03-03_ndmi.tif
[Line 8] ... processing :  s2_derived_tifs/14QQG_2025-03-08_ndmi.tif
[Line 8] ... processing :  s2_derived_tifs/14QQG_2025-03-13_ndmi.tif
[Line 8] ... processing :  s2_derived_tifs/14QQG_2025-03-18_ndmi.tif
[Line 8] ... processing :  s2_derived_tifs/14QQG_2025-03-23_ndmi.tif
[Line 8] ... processing :  s2_derived_tifs/14QQG_2025-03-28_ndmi.tif
[Line 8] ... processing :  s2_derived_tifs/14QQG_2025-04-02_ndmi.tif
[Line 8] ... processing :  s2_derived_tifs/14QQG_2025-04-07_ndmi.tif
[Line 8] ... processing :  s2_derived_tifs/14QQG_2025-04-12_ndmi.tif
[Line 8] ... processing :  s2_derived_tifs/14QQG_2025-04-17_ndmi.tif
[Line 8] ... processing :  s2_derived_tifs/14QQG_2025-04-22_ndmi.tif
[Line 8] ... processing :  s2_derived_tifs/14QQG_2025-04-27_ndmi.tif
[Line 8] ... processing :  s2_derived_tifs/14QQG_2025-05-02_ndmi.tif
[Line 8] ... processing :  s2_derived_tifs/14QQG_2025-05-07_ndmi.tif
[Line 8] ... processing :  s2_deri

In [20]:

import leafmap
import importlib
#import ConstantObjects
importlib.reload(ConstantObjects)
m2 = leafmap.Map(center=(Constants.lat0,Constants.lon0), zoom=22)
#gndvi_tif = Utils.make_path_name("s2_derived_tifs", Utils.latlon_to_s2_tile(Constants.lat0, Constants.lon0),date,"gndvi","tif"   ) #"ndmi_fixed.tif"

m2.add_basemap("Esri.WorldImagery")  # Google satellite imagery
m2.add_raster(output_file, layer_name="ndmi", colormap="BrBG", nodata=np.nan, opacity = .9)

print(f"[Cell Line {inspect.currentframe().f_lineno}] .. ")

print(f"[Cell Line {inspect.currentframe().f_lineno}] .. ")

m2.add_gdf(ConstantObjects.gdf_box_milpa_vecino, layer_name="milpa vecino")

m2.add_gdf(ConstantObjects.gdf_box_plots_4_5, layer_name="Plots 4,5")
m2.add_gdf(ConstantObjects.gdf_box_plots_2_3_1A_1B , layer_name="Plots 2,3,1A,1B")
m2.add_gdf(ConstantObjects.gdf_plotMilpa1 , layer_name="Milpa1")

m2.center=(Constants.lat0,Constants.lon0)
m2.zoom = 18

m2
#The following did not work well:
#m2.to_image(outfile='ndmi_mean.on-map.png')
#m2.to_image(outfile='ndmi_mean.on-map.jpg')

[Line 13] n_cols in ConstantObjects: 21
[Line 15] n_cols: 21, n_rows: 18
[Line 26] n_cols in ConstantObjects: 21
[Line 28] gdf_tree_circles.shape: (378, 1)
x0 =  -10783454.50783975
y0 =  2246774.099946275
draw_grid_box : dx =  5
draw_grid_box : dy =  4
📍 Center of box: (19.777898, -96.869939)
x0 =  -10783454.50783975
y0 =  2246774.099946275
draw_grid_box : dx =  5
draw_grid_box : dy =  4
📍 Center of box: (19.777882, -96.869634)
x0 =  -10783576.959279623
y0 =  2246703.1213188255
draw_grid_box : dx =  5
draw_grid_box : dy =  4
📍 Center of box: (19.777201, -96.871090)
Constants.n_cols =  21
Constants.n_rows =  18
x0 =  -10783454.50783975
y0 =  2246395.5503548854
draw_grid_box : dx =  5
draw_grid_box : dy =  4
📍 Center of box: (19.778652, -96.863761)
x0 =  -10783454.50783975
y0 =  2246774.099946275
draw_grid_box : dx =  5
draw_grid_box : dy =  4
📍 Center of box: (19.777761, -96.869698)
x0 =  -10783398.848094355
y0 =  2246927.887889348
draw_grid_box : dx =  5
draw_grid_box : dy =  4
📍 Cente

Map(center=[19.7782, -96.86942], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', '…